# Notebook 8: SOTA-Chasing Engagement Model (4-Class)

This notebook is a benchmark-focused pipeline for DAiSEE Engagement only (4 classes).
It is designed to maximize leaderboard-style engagement accuracy, not multi-dimension coverage.

## Key design
- Engagement-only 4-class training
- Dual-stream input: face crop + context crop
- EfficientNet backbones + temporal BiLSTM attention
- Class-balanced focal loss + weighted sampler
- Freeze -> unfreeze schedule + AMP + checkpoint/resume
- Test-time augmentation

## Important
No script can guarantee >73% on every run. This notebook is optimized to chase that target with strong defaults and reproducibility.

In [ ]:
# Cell 2: Optional installs (Kaggle)
# If already installed, this cell is safe to skip.
!pip -q install facenet-pytorch==2.6.0 --no-deps

In [ ]:
# Cell 3: Imports and config
import os
import gc
import re
import cv2
import json
import glob
import math
import time
import random
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import torchvision.transforms as T
from torchvision.models import (
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
    efficientnet_b0, EfficientNet_B0_Weights
)

from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

warnings.filterwarnings('ignore')

try:
    from facenet_pytorch import MTCNN
    MTCNN_AVAILABLE = True
except Exception:
    MTCNN_AVAILABLE = False

NOTEBOOK_VERSION = 'sota73_dualstream_v3_idmatch_20260317'
print('Notebook version:', NOTEBOOK_VERSION)

class CFG:
    SEED = 42

    # Paths
    DAISEE_ROOT = '/kaggle/input/datasets/seversnape/daisee-videos/DAiSEE'
    VIDEO_DIR = None
    LABELS_DIR = None

    OUT_DIR = '/kaggle/working'
    MODEL_DIR = '/kaggle/working/trained_models'
    RESULT_DIR = '/kaggle/working/experiment_results'
    CKPT_DIR = '/kaggle/working/checkpoints_sota73'
    FACE_CACHE = '/kaggle/working/face_bbox_cache_engagement.pkl'

    # Task
    DIMENSION = 'engagement'
    DIM_INDEX = 1  # boredom=0, engagement=1, confusion=2, frustration=3
    NUM_CLASSES = 4

    # Video
    N_FRAMES = 32
    IMG_SIZE = 224
    FACE_PAD_RATIO = 0.30
    FACE_DETECT_FRAMES = 4
    FACE_MIN_CONF = 0.80

    # Train
    BATCH_SIZE = 4
    NUM_WORKERS = 2
    EPOCHS = 24
    FREEZE_EPOCHS = 3
    LR_HEAD = 3e-4
    LR_BACKBONE = 5e-5
    WEIGHT_DECAY = 1e-4
    PATIENCE = 6
    LABEL_SMOOTHING = 0.05
    FOCAL_GAMMA = 2.0

    # Resume
    RESUME = True
    RUN_TAG = 'engagement_sota73_dualstream'

    # TTA
    TTA_FLIPS = True

os.makedirs(CFG.MODEL_DIR, exist_ok=True)
os.makedirs(CFG.RESULT_DIR, exist_ok=True)
os.makedirs(CFG.CKPT_DIR, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print('  GPU', i, torch.cuda.get_device_name(i))

In [ ]:
# Cell 4: Dataset discovery + labels (Engagement only)
def find_dataset_paths(root):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f'DAiSEE root not found: {root}')
    video_dir = root / 'DataSet'
    labels_dir = root / 'Labels'
    if not video_dir.exists() or not labels_dir.exists():
        raise FileNotFoundError('Could not find DataSet/ and Labels/ under DAiSEE root')
    return str(video_dir), str(labels_dir)

def normalize_clip_id(x):
    s = str(x).strip().strip('"').strip("'")
    s = s.replace('\\', '/').split('/')[-1].strip()
    lower = s.lower()
    for ext in ('.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.mpg', '.wmv'):
        if lower.endswith(ext):
            s = s[: -len(ext)]
            break
    # Handle pandas float-like ids such as '12345.0'
    if s.endswith('.0'):
        prefix = s[:-2]
        if prefix.isdigit():
            s = prefix
    return s

def alnum_key(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

def digit_key(s):
    return ''.join(ch for ch in str(s) if ch.isdigit())

def possible_keys(raw_clip_id):
    k = normalize_clip_id(raw_clip_id)
    out = [k, k.lower(), k.replace('_', ''), k.replace('-', ''), k.replace(' ', '')]

    # Numeric variants
    d = digit_key(k)
    if d:
        out.append(d)
        out.append(d.lstrip('0') or '0')

    # Alnum canonical variant
    a = alnum_key(k)
    if a:
        out.append(a)

    # Deduplicate while preserving order
    seen = set()
    uniq = []
    for x in out:
        if x and x not in seen:
            seen.add(x)
            uniq.append(x)
    return uniq

CFG.VIDEO_DIR, CFG.LABELS_DIR = find_dataset_paths(CFG.DAISEE_ROOT)
print('Video dir:', CFG.VIDEO_DIR)
print('Labels dir:', CFG.LABELS_DIR)

def load_labels_split(csv_path):
    # Read clip ids as strings to avoid accidental numeric coercion
    df = pd.read_csv(csv_path, dtype=str).fillna('')
    cols = {c.lower().strip(): c for c in df.columns}
    clip_col = cols.get('clipid')
    if clip_col is None:
        # fallback for any column containing 'clip'
        clip_candidates = [c for c in df.columns if 'clip' in c.lower()]
        if not clip_candidates:
            raise ValueError(f'Could not find clip-id column in {csv_path}')
        clip_col = clip_candidates[0]

    def find_col(prefix):
        for c in df.columns:
            if c.lower().strip().startswith(prefix):
                return c
        raise ValueError(f'Missing required label column starting with {prefix} in {csv_path}')

    b_col = find_col('boredom')
    e_col = find_col('engagement')
    c_col = find_col('confusion')
    f_col = find_col('frustration')

    rows = []
    for _, r in df.iterrows():
        clip_id = normalize_clip_id(r[clip_col])
        labels = [int(float(r[b_col])), int(float(r[e_col])), int(float(r[c_col])), int(float(r[f_col]))]
        rows.append((clip_id, labels))
    return rows

train_rows = load_labels_split(os.path.join(CFG.LABELS_DIR, 'TrainLabels.csv'))
val_rows = load_labels_split(os.path.join(CFG.LABELS_DIR, 'ValidationLabels.csv'))
test_rows = load_labels_split(os.path.join(CFG.LABELS_DIR, 'TestLabels.csv'))
print('Rows loaded:', len(train_rows), len(val_rows), len(test_rows))

# Build multiple lookup indices from video files
allowed_ext = {'.mp4', '.avi', '.mov', '.mkv', '.mpeg', '.mpg', '.wmv'}
video_exact = {}
video_alnum = {}
video_digits = {}
video_paths = []

for root, _, files in os.walk(CFG.VIDEO_DIR):
    for fn in files:
        ext = Path(fn).suffix.lower()
        if ext not in allowed_ext:
            continue
        full = os.path.join(root, fn)
        video_paths.append(full)

        stem = normalize_clip_id(fn)
        keys = possible_keys(stem)
        for key in keys:
            video_exact.setdefault(key, full)

        ak = alnum_key(stem)
        if ak:
            video_alnum.setdefault(ak, full)

        dk = digit_key(stem)
        if dk:
            video_digits.setdefault(dk, full)
            video_digits.setdefault(dk.lstrip('0') or '0', full)

print('Indexed video files:', len(video_paths))
print('Unique exact keys:', len(video_exact))
print('Unique alnum keys:', len(video_alnum))
print('Unique digit keys:', len(video_digits))
print('Sample exact keys:', list(video_exact.keys())[:5])
print('Sample label keys:', [x[0] for x in train_rows[:5]])

def resolve_video_path(clip_id):
    # 1) exact-style keys
    for k in possible_keys(clip_id):
        p = video_exact.get(k)
        if p is not None:
            return p

    # 2) alnum canonical key
    ak = alnum_key(clip_id)
    if ak and ak in video_alnum:
        return video_alnum[ak]

    # 3) digits-only key
    dk = digit_key(clip_id)
    if dk:
        p = video_digits.get(dk) or video_digits.get(dk.lstrip('0') or '0')
        if p is not None:
            return p

    # 4) last resort: substring search in path alnum
    # This is slower but robust for oddly formatted ids.
    if ak:
        for p in video_paths:
            if ak in alnum_key(p):
                return p
    return None

def build_split(rows):
    out = []
    miss = 0
    misses = []
    for cid, labs in rows:
        vp = resolve_video_path(cid)
        if vp is None:
            miss += 1
            if len(misses) < 12:
                misses.append(cid)
            continue
        out.append({'clip_id': cid, 'video_path': vp, 'label4': int(labs[CFG.DIM_INDEX])})
    return out, miss, misses

train_data, miss_tr, miss_keys_tr = build_split(train_rows)
val_data, miss_va, miss_keys_va = build_split(val_rows)
test_data, miss_te, miss_keys_te = build_split(test_rows)
print('Matched:', len(train_data), len(val_data), len(test_data))
print('Missing:', miss_tr, miss_va, miss_te)
if miss_tr > 0:
    print('Example unmatched train ids:', miss_keys_tr)

def show_dist(name, items):
    y = [x['label4'] for x in items]
    cnt = dict(sorted(Counter(y).items()))
    print(name, cnt)

show_dist('Train engagement distribution:', train_data)
show_dist('Val engagement distribution:', val_data)
show_dist('Test engagement distribution:', test_data)

if len(train_data) == 0 or len(val_data) == 0 or len(test_data) == 0:
    raise RuntimeError(
        'One or more splits are empty after matching. Confirm you uploaded THIS v3 notebook and check sample keys above.'
    )

In [ ]:
# Cell 5: Face bbox cache using MTCNN
def sample_frame_indices(total, n):
    if total <= 0:
        return [0] * n
    if total <= n:
        idx = list(range(total))
        while len(idx) < n:
            idx.append(idx[-1])
        return idx
    return np.linspace(0, total - 1, n).astype(int).tolist()

def detect_face_bbox(video_path, mtcnn=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    check_idx = sample_frame_indices(total, CFG.FACE_DETECT_FRAMES)
    best = None
    best_score = -1
    for fi in check_idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        if mtcnn is None:
            h, w = rgb.shape[:2]
            return (int(0.2 * w), int(0.15 * h), int(0.8 * w), int(0.9 * h))
        boxes, probs = mtcnn.detect(Image.fromarray(rgb))
        if boxes is None:
            continue
        for b, p in zip(boxes, probs):
            if p is None or p < CFG.FACE_MIN_CONF:
                continue
            x1, y1, x2, y2 = b.tolist()
            area = max(0, x2 - x1) * max(0, y2 - y1)
            score = float(p) * area
            if score > best_score:
                best_score = score
                best = (int(x1), int(y1), int(x2), int(y2))
    cap.release()
    return best

if os.path.exists(CFG.FACE_CACHE):
    with open(CFG.FACE_CACHE, 'rb') as f:
        face_cache = pickle.load(f)
    print('Loaded face cache:', len(face_cache))
else:
    face_cache = {}

all_items = train_data + val_data + test_data
pending = [x for x in all_items if x['clip_id'] not in face_cache]
print('Pending face detections:', len(pending))

if len(pending) > 0:
    mtcnn = None
    if MTCNN_AVAILABLE:
        mtcnn = MTCNN(keep_all=True, device=DEVICE)
    for item in tqdm(pending, desc='Face detection cache'):
        bbox = detect_face_bbox(item['video_path'], mtcnn)
        face_cache[item['clip_id']] = bbox
    with open(CFG.FACE_CACHE, 'wb') as f:
        pickle.dump(face_cache, f)
    print('Face cache saved:', CFG.FACE_CACHE)

detected = sum(v is not None for v in face_cache.values())
print('Detected faces:', detected, '/', len(face_cache))

In [ ]:
# Cell 6: Dataset and model
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.ToTensor(),
    T.Normalize(imagenet_mean, imagenet_std),
])

test_tf = T.Compose([
    T.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(imagenet_mean, imagenet_std),
])

def crop_face(rgb, bbox):
    h, w = rgb.shape[:2]
    if bbox is None:
        x1, y1, x2, y2 = int(0.2 * w), int(0.15 * h), int(0.8 * w), int(0.9 * h)
    else:
        x1, y1, x2, y2 = bbox
        bw = x2 - x1
        bh = y2 - y1
        padw = int(CFG.FACE_PAD_RATIO * bw)
        padh = int(CFG.FACE_PAD_RATIO * bh)
        x1 = max(0, x1 - padw)
        y1 = max(0, y1 - padh)
        x2 = min(w - 1, x2 + padw)
        y2 = min(h - 1, y2 + padh)
    if x2 <= x1 or y2 <= y1:
        return rgb
    return rgb[y1:y2, x1:x2]

class DAiSEEEngagementDataset(Dataset):
    def __init__(self, items, face_cache, transform, n_frames=32):
        self.items = items
        self.face_cache = face_cache
        self.transform = transform
        self.n_frames = n_frames

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        cap = cv2.VideoCapture(item['video_path'])
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_idx = sample_frame_indices(total, self.n_frames)
        bbox = self.face_cache.get(item['clip_id'])

        face_frames, ctx_frames = [], []
        for fi in frame_idx:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok or frame is None:
                frame = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            face = crop_face(rgb, bbox)
            ctx = rgb
            face_frames.append(self.transform(Image.fromarray(face)))
            ctx_frames.append(self.transform(Image.fromarray(ctx)))
        cap.release()

        face_tensor = torch.stack(face_frames, dim=0)  # (T,C,H,W)
        ctx_tensor = torch.stack(ctx_frames, dim=0)    # (T,C,H,W)
        label = torch.tensor(item['label4'], dtype=torch.long)
        return face_tensor, ctx_tensor, label

class EfficientFrameEncoder(nn.Module):
    def __init__(self, kind='v2s'):
        super().__init__()
        if kind == 'v2s':
            base = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
            feat_dim = 1280
        else:
            base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
            feat_dim = 1280
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.feat_dim = feat_dim

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return x

class TemporalHead(nn.Module):
    def __init__(self, in_dim, hidden=384, num_classes=4, drop=0.4):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.lstm = nn.LSTM(hidden, hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=drop)
        self.attn = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.Tanh(), nn.Linear(hidden, 1))
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, seq):
        z = self.proj(seq)
        out, _ = self.lstm(z)
        w = torch.softmax(self.attn(out).squeeze(-1), dim=1)
        pooled = (out * w.unsqueeze(-1)).sum(1)
        return self.fc(self.drop(pooled))

class DualStreamEngagementModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.face_enc = EfficientFrameEncoder('v2s')
        self.ctx_enc = EfficientFrameEncoder('b0')
        self.head = TemporalHead(1280 + 1280, hidden=384, num_classes=CFG.NUM_CLASSES, drop=0.4)

    def forward(self, face_seq, ctx_seq):
        # face_seq, ctx_seq: (B,T,C,H,W)
        b, t, c, h, w = face_seq.shape
        face = face_seq.view(b * t, c, h, w)
        ctx = ctx_seq.view(b * t, c, h, w)
        f1 = self.face_enc(face).view(b, t, -1)
        f2 = self.ctx_enc(ctx).view(b, t, -1)
        fused = torch.cat([f1, f2], dim=-1)
        return self.head(fused)

model = DualStreamEngagementModel().to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable params:', f'{n_params:,}')

In [ ]:
# Cell 7: Loss, loaders, train/eval loops
if len(train_data) == 0 or len(val_data) == 0 or len(test_data) == 0:
    raise RuntimeError('Empty data split detected before dataloader creation.')

def class_weights_from_data(items, num_classes=4):
    y = [x['label4'] for x in items]
    cnt = Counter(y)
    total = sum(cnt.values())
    w = []
    for c in range(num_classes):
        nc = max(cnt.get(c, 1), 1)
        w.append(total / (num_classes * nc))
    w = np.array(w, dtype=np.float32)
    w = w / w.sum() * num_classes
    return w

class FocalCE(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none', label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            a = self.alpha[target]
            loss = a * loss
        return loss.mean()

train_ds = DAiSEEEngagementDataset(train_data, face_cache, train_tf, n_frames=CFG.N_FRAMES)
val_ds = DAiSEEEngagementDataset(val_data, face_cache, test_tf, n_frames=CFG.N_FRAMES)
test_ds = DAiSEEEngagementDataset(test_data, face_cache, test_tf, n_frames=CFG.N_FRAMES)

# Weighted sampler
class_w = class_weights_from_data(train_data, CFG.NUM_CLASSES)
sample_w = np.array([class_w[x['label4']] for x in train_data], dtype=np.float64)
if len(sample_w) == 0:
    raise RuntimeError('sample_w is empty; train_data has no matched rows.')
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                          num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                        num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

alpha = torch.tensor(class_w, dtype=torch.float32, device=DEVICE)
criterion = FocalCE(alpha=alpha, gamma=CFG.FOCAL_GAMMA, label_smoothing=CFG.LABEL_SMOOTHING)

raw_model = model.module if hasattr(model, 'module') else model
optim = torch.optim.AdamW(
    [
        {'params': raw_model.head.parameters(), 'lr': CFG.LR_HEAD},
        {'params': raw_model.face_enc.parameters(), 'lr': CFG.LR_BACKBONE},
        {'params': raw_model.ctx_enc.parameters(), 'lr': CFG.LR_BACKBONE},
    ],
    weight_decay=CFG.WEIGHT_DECAY
)
sched = CosineAnnealingLR(optim, T_max=CFG.EPOCHS, eta_min=1e-6)
scaler = GradScaler(enabled=(DEVICE.type == 'cuda'))

def set_backbones_trainable(enabled):
    raw = model.module if hasattr(model, 'module') else model
    for p in raw.face_enc.parameters():
        p.requires_grad = enabled
    for p in raw.ctx_enc.parameters():
        p.requires_grad = enabled

def evaluate(loader, use_tta=False):
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for face, ctx, y in tqdm(loader, leave=False):
            face = face.to(DEVICE, non_blocking=True)
            ctx = ctx.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                logits = model(face, ctx)
                probs = torch.softmax(logits, dim=1)
                if use_tta and CFG.TTA_FLIPS:
                    f2 = torch.flip(face, dims=[4])
                    c2 = torch.flip(ctx, dims=[4])
                    logits2 = model(f2, c2)
                    probs2 = torch.softmax(logits2, dim=1)
                    probs = 0.5 * (probs + probs2)
            all_probs.append(probs.detach().cpu().numpy())
            all_true.append(y.detach().cpu().numpy())
    p = np.concatenate(all_probs)
    t = np.concatenate(all_true)
    pred = p.argmax(1)
    acc = accuracy_score(t, pred)
    f1m = f1_score(t, pred, average='macro', zero_division=0)
    qwk = cohen_kappa_score(t, pred, weights='quadratic')
    return {'acc': float(acc), 'f1m': float(f1m), 'qwk': float(qwk)}, p, t

In [ ]:
# Cell 8: Train with checkpoint/resume
ckpt_path = os.path.join(CFG.CKPT_DIR, f'{CFG.RUN_TAG}_last.pt')
best_path = os.path.join(CFG.CKPT_DIR, f'{CFG.RUN_TAG}_best.pt')

start_epoch = 0
best_val = -1.0
pat = 0

if CFG.RESUME and os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, map_location='cpu')
    (model.module if hasattr(model, 'module') else model).load_state_dict(ck['model'])
    optim.load_state_dict(ck['optim'])
    sched.load_state_dict(ck['sched'])
    scaler.load_state_dict(ck['scaler'])
    start_epoch = int(ck['epoch']) + 1
    best_val = float(ck.get('best_val', -1.0))
    pat = int(ck.get('pat', 0))
    print('Resumed from epoch', start_epoch)

for epoch in range(start_epoch, CFG.EPOCHS):
    # Freeze early, unfreeze later
    if epoch < CFG.FREEZE_EPOCHS:
        set_backbones_trainable(False)
    else:
        set_backbones_trainable(True)

    model.train()
    run_loss = 0.0
    y_true, y_pred = [], []

    for face, ctx, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{CFG.EPOCHS}'):
        face = face.to(DEVICE, non_blocking=True)
        ctx = ctx.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            logits = model(face, ctx)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim)
        scaler.update()

        run_loss += loss.item() * y.size(0)
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())

    sched.step()

    tr_acc = accuracy_score(y_true, y_pred)
    tr_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    tr_loss = run_loss / max(1, len(y_true))

    val_m, _, _ = evaluate(val_loader, use_tta=False)
    print(f'Epoch {epoch+1}: loss={tr_loss:.4f} tr_acc={tr_acc:.4f} tr_f1m={tr_f1:.4f} | val_acc={val_m["acc"]:.4f} val_f1m={val_m["f1m"]:.4f} val_qwk={val_m["qwk"]:.4f}')

    state = {
        'epoch': epoch,
        'model': (model.module if hasattr(model, 'module') else model).state_dict(),
        'optim': optim.state_dict(),
        'sched': sched.state_dict(),
        'scaler': scaler.state_dict(),
        'best_val': best_val,
        'pat': pat,
    }
    torch.save(state, ckpt_path)

    if val_m['acc'] > best_val:
        best_val = val_m['acc']
        pat = 0
        torch.save(state, best_path)
        print('  New best checkpoint saved.')
    else:
        pat += 1

    if pat >= CFG.PATIENCE:
        print('Early stopping triggered.')
        break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# Cell 9: Test evaluation + export outputs
if os.path.exists(best_path):
    best = torch.load(best_path, map_location='cpu')
    (model.module if hasattr(model, 'module') else model).load_state_dict(best['model'])
    print('Loaded best checkpoint from epoch', best['epoch'] + 1)

test_metrics, test_probs, test_true = evaluate(test_loader, use_tta=True)
pred = test_probs.argmax(1)
cm = confusion_matrix(test_true, pred, labels=[0, 1, 2, 3])

print('\n=== TEST RESULTS (Engagement 4-class) ===')
print('Accuracy:', f"{test_metrics['acc']:.4f}")
print('F1-macro:', f"{test_metrics['f1m']:.4f}")
print('QWK:', f"{test_metrics['qwk']:.4f}")
print('Confusion matrix:\n', cm)

# Save outputs for comparison and downstream fusion
np.save(os.path.join(CFG.MODEL_DIR, 'proba_sota73_4c_engagement.npy'), test_probs)
np.save(os.path.join(CFG.MODEL_DIR, 'labels_4class_engagement.npy'), test_true)
np.save(os.path.join(CFG.MODEL_DIR, 'proba_sota73_engagement.npy'), test_probs[:, 2:].sum(axis=1))
torch.save((model.module if hasattr(model, 'module') else model).state_dict(),
           os.path.join(CFG.MODEL_DIR, 'sota73_dualstream_engagement.pt'))

result = {
    'experiment': CFG.RUN_TAG,
    'task': 'engagement_4class',
    'n_frames': CFG.N_FRAMES,
    'test_accuracy': test_metrics['acc'],
    'test_f1_macro': test_metrics['f1m'],
    'test_qwk': test_metrics['qwk'],
    'checkpoint_best': best_path if os.path.exists(best_path) else None,
}
out_json = os.path.join(CFG.RESULT_DIR, f'{CFG.RUN_TAG}_results.json')
with open(out_json, 'w') as f:
    json.dump(result, f, indent=2)
print('Saved:', out_json)

print('\nComparison anchors:')
print('- CORAL baseline (repo): 48.3% engagement 4-class acc')
print('- Selim et al. 2022: 67.48%')
print('- ViBED-Net 2025: 73.43%')
print('- This run:', f"{test_metrics['acc'] * 100:.2f}%")

## Notes for pushing above 73%

1. Increase frame count to 40-48 with gradient accumulation if GPU memory allows.
2. Replace context encoder with EfficientNetV2-S (second stream) if memory permits.
3. Add pretraining on facial expression data before DAiSEE fine-tuning.
4. Run 3-5 seeds and ensemble logits for final report.
5. Keep the official test split untouched for fair SOTA comparison.